# Semantic Relational Atlas — real-model run

This notebook runs the hardened pilot on **real pretrained weights**:

- `Qwen/Qwen2.5-0.5B`
- `bigscience/bloom-560m`
- `facebook/xglm-564M`

It extracts layer-wise hidden states for matched English/Chinese concepts, chooses layers on a **selection split**, and evaluates convergence on held-out concepts with permutation nulls, bootstrap intervals, and FDR correction.

### Before running

Use a Colab GPU runtime (`Runtime → Change runtime type → T4 GPU` or better).

The notebook downloads several GB of official model weights. Models are loaded one at a time.


In [ ]:
# 1) Environment check
import os, sys, subprocess, platform
try:
    import torch
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
    import torch

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "A GPU runtime is strongly recommended. In Colab choose "
        "Runtime → Change runtime type → T4 GPU, then run all cells again."
    )


In [ ]:
# 2) Install the model/runtime dependencies
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==5.16.1",
    "tokenizers>=0.22.0",
    "sentencepiece>=0.2",
    "safetensors>=0.4",
    "numpy>=1.26",
    "pandas>=2.0",
    "scipy>=1.11",
    "plotly>=5.20",
], check=True)
print("Dependencies installed.")


In [ ]:
# 3) Unpack the complete hardened experiment embedded in this notebook
# Regenerate with scripts/build_notebook.py -- do not hand-edit.
import base64, io, zipfile, os
payload = """..."""
project_dir = "/content/semantic_relational_atlas"
os.makedirs(project_dir, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as z:
    z.extractall(project_dir)
print("Project unpacked to:", project_dir)
print("Files:", sorted(os.listdir(project_dir)))


In [ ]:
# 4) Show the exact benchmark this run will use
from pathlib import Path
import json, csv

project = Path("/content/semantic_relational_atlas")

with open(project/"benchmark"/"concepts.csv", encoding="utf-8") as f:
    concepts = list(csv.DictReader(f))
with open(project/"benchmark"/"relations.csv", encoding="utf-8") as f:
    relations = list(csv.DictReader(f))
cfg = json.loads((project/"experiment_config.json").read_text())

# Every relation type must have all-distinct targets. A relation that reuses a
# target cannot be tested this way: the correct multiplicity-preserving null
# converges on the observed data, so the test has no power by construction.
import collections
reuse = collections.Counter()
for r in relations:
    reuse[r["relation"]] += 1
distinct = collections.defaultdict(set)
for r in relations:
    distinct[r["relation"]].add(r["target"])
bad = {k: reuse[k]/len(distinct[k]) for k in reuse if reuse[k] != len(distinct[k])}
assert not bad, f"relations reuse targets and are not testable: {bad}"

print("Concepts:", len(concepts))
print("Relation probes:", len(relations), "across", len(reuse), "types, all targets distinct")
print("Models:")
for k,v in cfg["models"].items():
    print(" ", k, "->", v)
print("Languages:", cfg["languages"])


## Run the real extraction

This is the expensive step. It loads each model sequentially and saves hidden-state representations for both English and Chinese.


In [ ]:
# 5) Extract real hidden-state representations
import subprocess, sys, os, time
project = "/content/semantic_relational_atlas"
env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"] = "false"
env["HF_HUB_DISABLE_TELEMETRY"] = "1"

t0 = time.time()
subprocess.run([
    sys.executable,
    "src/extract_representations.py",
    "--device", "cuda",
    "--prompt-mode", "bare",
], cwd=project, env=env, check=True)
print("Extraction complete. Minutes:", round((time.time()-t0)/60, 1))


## Hardened statistical analysis

The default below uses **1,000 permutations and 2,000 bootstrap replicates**. For a quick exploratory run, change these to 100 and 200.


In [ ]:
# 6) Run held-out geometry + relation tests
import subprocess, sys, time
project = "/content/semantic_relational_atlas"

PERMUTATIONS = 1000
BOOTSTRAPS = 2000
SEED = 20260903

t0 = time.time()
subprocess.run([
    sys.executable,
    "src/analyze_semantic_geometry.py",
    "--prompt-mode", "bare",
    "--permutations", str(PERMUTATIONS),
    "--bootstrap", str(BOOTSTRAPS),
    "--seed", str(SEED),
], cwd=project, check=True)
print("Analysis complete. Minutes:", round((time.time()-t0)/60, 1))


In [ ]:
# 7) Generate the interactive HTML report
import subprocess, sys
project = "/content/semantic_relational_atlas"
subprocess.run([sys.executable, "src/make_report.py"], cwd=project, check=True)
print("Report:", project + "/outputs/semantic_relational_report.html")


## Read the primary results

These tables—not the 2-D map—are the core outcome.


In [ ]:
# 8) Display headline results
from pathlib import Path
import json, pandas as pd
from IPython.display import display

out = Path("/content/semantic_relational_atlas/outputs/analysis")
summary = json.loads((out/"summary.json").read_text())

print("HELD-OUT NEIGHBORHOOD TEST")
display(pd.read_csv(out/"neighborhood_null_test.csv"))

print("\nHELD-OUT SYSTEM ALIGNMENTS")
a = pd.read_csv(out/"system_alignment.csv")
display(a.sort_values("heldout_spearman_rho", ascending=False))

print("\nRELATION CONVERGENCE — hardest (region-matched) null")
r = pd.read_csv(out/"relation_convergence.csv")
cols = [
    "relation","n_pairs",
    "heldout_anchor_cross_system_signature_similarity",
    "bootstrap_ci_lo","bootstrap_ci_hi",
    "region_matched_null_mean",
    "region_matched_effect_over_null",
    "region_matched_permutation_p",
    "region_matched_fdr_q"
]
display(r[cols].sort_values(
    "heldout_anchor_cross_system_signature_similarity",
    ascending=False
))

print("\nSUMMARY")
print(json.dumps(summary, indent=2))


In [ ]:
# 9) A conservative automatic interpretation
import pandas as pd
from pathlib import Path

out = Path("/content/semantic_relational_atlas/outputs/analysis")
n = pd.read_csv(out/"neighborhood_null_test.csv").iloc[0]
r = pd.read_csv(out/"relation_convergence.csv")

print("=== PRELIMINARY INTERPRETATION ===")
if n["permutation_p"] <= 0.05 and n["effect_over_null"] > 0:
    print("✓ Held-out semantic neighborhoods converge across systems above the permutation null.")
else:
    print("✗ No clear held-out neighborhood-convergence result at p <= .05 in this run.")

sig = r[
    (r["region_matched_fdr_q"] <= 0.05) &
    (r["region_matched_effect_over_null"] > 0)
].sort_values("region_matched_effect_over_null", ascending=False)

if len(sig):
    print("\nRelation types surviving the harder region-matched null + FDR:")
    for _,row in sig.iterrows():
        print(
            f"  ✓ {row['relation']}: "
            f"similarity={row['heldout_anchor_cross_system_signature_similarity']:.3f}, "
            f"effect={row['region_matched_effect_over_null']:.3f}, "
            f"q={row['region_matched_fdr_q']:.4g}"
        )
else:
    print("\nNo relation type survives the harder region-matched null + FDR in this pilot.")

print("\nThis is exploratory evidence only; isolated-word prompts and the small curated benchmark remain major limitations.")


In [ ]:
# 10) Download the result bundle from Colab
from pathlib import Path
import shutil
from google.colab import files

project = Path("/content/semantic_relational_atlas")
archive = shutil.make_archive(
    "/content/semantic_relational_atlas_REAL_RESULTS",
    "zip",
    root_dir=project,
    base_dir="outputs"
)
print("Created:", archive)
files.download(archive)
